In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(r"c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest")

MERGED_DIR = PROJECT_DIR / "MERGED_DATASET"

OUTPUT_DIR = PROJECT_DIR / "PREPROCESSED_DATASET"

OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("Merged Dataset :", MERGED_DIR)
print("Output Folder  :", OUTPUT_DIR)
print("=" * 60)

Merged Dataset : c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\MERGED_DATASET
Output Folder  : c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\PREPROCESSED_DATASET


In [3]:
# ============================================================
# LOAD ALL MERGED FILES
# ============================================================

merged_files = sorted(MERGED_DIR.glob("merged_*.parquet"))

print(f"Total merged files : {len(merged_files)}")

print("\nFirst five files:")

for f in merged_files[:5]:
    print(f.name)

Total merged files : 728

First five files:
merged_20240201.parquet
merged_20240202.parquet
merged_20240203.parquet
merged_20240204.parquet
merged_20240205.parquet


In [4]:
# ============================================================
# LOAD ONE SAMPLE FILE
# ============================================================

sample_file = merged_files[0]

df = pd.read_parquet(sample_file)

print("Loaded:", sample_file.name)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

Loaded: merged_20240201.parquet

Shape:
(86394, 346)

Columns:
['unix_time', 'lc_counts_scaled', 'hardness_ratio', 'ch_000', 'ch_001', 'ch_002', 'ch_003', 'ch_004', 'ch_005', 'ch_006', 'ch_007', 'ch_008', 'ch_009', 'ch_010', 'ch_011', 'ch_012', 'ch_013', 'ch_014', 'ch_015', 'ch_016', 'ch_017', 'ch_018', 'ch_019', 'ch_020', 'ch_021', 'ch_022', 'ch_023', 'ch_024', 'ch_025', 'ch_026', 'ch_027', 'ch_028', 'ch_029', 'ch_030', 'ch_031', 'ch_032', 'ch_033', 'ch_034', 'ch_035', 'ch_036', 'ch_037', 'ch_038', 'ch_039', 'ch_040', 'ch_041', 'ch_042', 'ch_043', 'ch_044', 'ch_045', 'ch_046', 'ch_047', 'ch_048', 'ch_049', 'ch_050', 'ch_051', 'ch_052', 'ch_053', 'ch_054', 'ch_055', 'ch_056', 'ch_057', 'ch_058', 'ch_059', 'ch_060', 'ch_061', 'ch_062', 'ch_063', 'ch_064', 'ch_065', 'ch_066', 'ch_067', 'ch_068', 'ch_069', 'ch_070', 'ch_071', 'ch_072', 'ch_073', 'ch_074', 'ch_075', 'ch_076', 'ch_077', 'ch_078', 'ch_079', 'ch_080', 'ch_081', 'ch_082', 'ch_083', 'ch_084', 'ch_085', 'ch_086', 'ch_087', 'ch_0

In [6]:
# ============================================================
# EVENT PARSER
# ============================================================

def parse_flare_regions(df):
    """
    Parse flare regions from the status column.

    Returns
    -------
    region1_pairs : list of tuples
        (start_index, peak_index)

    region2_pairs : list of tuples
        (last_peak_index, end_index)
    """

    region1_pairs = []
    region2_pairs = []

    current_start = None
    last_peak = None

    for idx, status in enumerate(df["status"]):

        # ----------------------------------------------------
        # EVENT_START
        # ----------------------------------------------------
        if status == "EVENT_START":

            # Ignore duplicate STARTs
            if current_start is None:
                current_start = idx

        # ----------------------------------------------------
        # EVENT_PEAK
        # ----------------------------------------------------
        elif status == "EVENT_PEAK":

            if current_start is not None:

                region1_pairs.append((current_start, idx))

                last_peak = idx
                current_start = None

        # ----------------------------------------------------
        # EVENT_END
        # ----------------------------------------------------
        elif status == "EVENT_END":

            if last_peak is not None:

                region2_pairs.append((last_peak, idx))

                last_peak = None

    return region1_pairs, region2_pairs

In [9]:
# ============================================================
# REGION 1 INTERPOLATION
# ============================================================

def fill_region1(df, region1_pairs):
    """
    Fill Region 1 (EVENT_START -> EVENT_PEAK).

    Weight:
        (LC(t) - LC(start)) /
        (LC(peak) - LC(start))

    xrsb(t):
        xrsb(start) +
        weight * (xrsb(peak) - xrsb(start))

    Only fills NaN xrsb_flux values.
    """

    df = df.copy()

    for start_idx, peak_idx in region1_pairs:

        # Boundary values
        lc_start = df.at[start_idx, "lc_counts_scaled"]
        lc_peak  = df.at[peak_idx, "lc_counts_scaled"]

        xrsb_start = df.at[start_idx, "xrsb_flux"]
        xrsb_peak  = df.at[peak_idx, "xrsb_flux"]

        # Safety checks
        if pd.isna(lc_start) or pd.isna(lc_peak):
            continue

        if pd.isna(xrsb_start) or pd.isna(xrsb_peak):
            continue

        denominator = lc_peak - lc_start

        if abs(denominator) < 1e-12:
            continue

        # ----------------------------------------------------
        # Fill every row BETWEEN start and peak
        # ----------------------------------------------------

        for idx in range(start_idx + 1, peak_idx):

            # Only fill missing values
            if pd.notna(df.at[idx, "xrsb_flux"]):
                continue

            lc_t = df.at[idx, "lc_counts_scaled"]

            if pd.isna(lc_t):
                continue

            weight = (lc_t - lc_start) / denominator

            xrsb_t = (
                xrsb_start
                + weight * (xrsb_peak - xrsb_start)
            )

            df.at[idx, "xrsb_flux"] = xrsb_t

    return df

In [10]:
# ============================================================
# APPLY REGION 1
# ============================================================

df_region1 = fill_region1(df, region1)

print("Region 1 interpolation completed.")

Region 1 interpolation completed.


In [12]:
# ============================================================
# REGION 2 INTERPOLATION
# ============================================================

def fill_region2(df, region2_pairs):
    """
    Fill Region 2 (EVENT_PEAK -> EVENT_END).

    Weight:
        (LC(peak) - LC(t)) /
        (LC(peak) - LC(end))

    xrsb(t):
        xrsb(peak) -
        weight * (xrsb(peak) - xrsb(end))

    Only fills NaN xrsb_flux values.
    """

    df = df.copy()

    for peak_idx, end_idx in region2_pairs:

        # Boundary values
        lc_peak = df.at[peak_idx, "lc_counts_scaled"]
        lc_end  = df.at[end_idx, "lc_counts_scaled"]

        xrsb_peak = df.at[peak_idx, "xrsb_flux"]
        xrsb_end  = df.at[end_idx, "xrsb_flux"]

        # Safety checks
        if pd.isna(lc_peak) or pd.isna(lc_end):
            continue

        if pd.isna(xrsb_peak) or pd.isna(xrsb_end):
            continue

        denominator = lc_peak - lc_end

        if abs(denominator) < 1e-12:
            continue

        # ----------------------------------------------------
        # Fill every row BETWEEN peak and end
        # ----------------------------------------------------

        for idx in range(peak_idx + 1, end_idx):

            # Only fill missing values
            if pd.notna(df.at[idx, "xrsb_flux"]):
                continue

            lc_t = df.at[idx, "lc_counts_scaled"]

            if pd.isna(lc_t):
                continue

            weight = (lc_peak - lc_t) / denominator

            xrsb_t = (
                xrsb_peak
                - weight * (xrsb_peak - xrsb_end)
            )

            df.at[idx, "xrsb_flux"] = xrsb_t

    return df

In [13]:
# ============================================================
# APPLY REGION 2
# ============================================================

df_region12 = fill_region2(
    df_region1,
    region2
)

print("Region 1 + Region 2 interpolation completed.")

Region 1 + Region 2 interpolation completed.


In [16]:
# ============================================================
# REMOVE REGION 0 (rows with unknown xrsb_flux)
# ============================================================

rows_before = len(df_region12)

df_final = (
    df_region12
    .dropna(subset=["xrsb_flux"])
    .reset_index(drop=True)
)

rows_after = len(df_final)

print("=" * 60)
print("Region 0 removed")
print("=" * 60)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Remaining NaNs in xrsb_flux : {df_final['xrsb_flux'].isna().sum()}")

Region 0 removed
Rows before : 86,394
Rows after  : 72,715
Rows removed: 13,679
Remaining NaNs in xrsb_flux : 0


In [17]:
# ============================================================
# FINAL CLEANUP
# ============================================================

df_final = df_final.drop(
    columns=[
        "status",
        "flare_class"
    ]
)

print("Columns removed successfully.")

Columns removed successfully.


In [ ]:
# ============================================================
# ENFORCE MINIMUM BACKGROUND FLUX
# ============================================================

# Use the background flux computed for this GOES file
background_flux_avg = df_final["xrsb_flux"].min()

# Replace values below the background with the background
df_final.loc[
    df_final["xrsb_flux"] < background_flux_avg,
    "xrsb_flux"
] = background_flux_avg

print("=" * 60)
print("Minimum background flux enforced")
print("=" * 60)
print(f"Background flux : {background_flux_avg:.6e}")

print(
    "Values below background :",
    (df_final["xrsb_flux"] < background_flux_avg).sum()
)

Minimum background flux enforced
Background flux : 8.539824e-07
Values below background : 0
